# SalesMesh Analytics Dashboard — Score New Leads

Okeyo Hoppe Solini Marie (161031) · Supervisor: Mr. Tiberius Tabulu · Strathmore University

This notebook demonstrates the system's core workflow from **input → processing → output**:

1. **Input:** a CSV file of new leads, in the same column format as the Kaggle Lead Scoring dataset
2. **Validation:** every row is checked, and rows with missing, invalid or unknown values are rejected with a reason
3. **Processing:** valid leads are encoded and scored with the trained model (`salesmesh_model.pkl`) and given a High / Medium / Low priority tier (FR-04)
4. **Output:** a SQL script that inserts the new leads and their predictions into the `salesmesh` MySQL database. After a refresh, they appear in the Power BI dashboard.

It also runs the **automated test cases** for this workflow and records each one as test / expected result / actual result / status.

**How to run:** Runtime → Change runtime type → **CPU**, then Runtime → Run all. You will be asked to upload two files:
- `salesmesh_model.pkl` (from the `models/` folder)
- `leads_with_predictions.csv` (from the `predictions/` folder)

The notebook does no model training, so it finishes in under a minute.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import joblib
import sklearn
from google.colab import files

print("scikit-learn version:", sklearn.__version__)

## 2. Upload the trained model and the reference data
- **`salesmesh_model.pkl`**: the selected Gradient Boosting pipeline (Min-Max scaler → SMOTE → model). SMOTE is only used while training, so it has no effect when scoring new leads.
- **`leads_with_predictions.csv`**: the 9,240 existing leads. They are used to check which category values the model knows, and which Prospect IDs already exist in the database.

In [ ]:
print("Upload: salesmesh_model.pkl")
uploaded = files.upload()
model = joblib.load(list(uploaded.keys())[0])

print("Upload: leads_with_predictions.csv")
uploaded = files.upload()
reference = pd.read_csv(list(uploaded.keys())[0])

print("Model steps:", [name for name, _ in model.steps])
print("Reference leads:", len(reference))

## 3. Define the input format and validation rules
A new-leads file must contain a `Prospect ID` and the 14 lead attributes the model was trained on, using the same column names as the original dataset.

| Rule | Applies to | Rejected when |
|---|---|---|
| Required file columns | Whole file | Any required column is missing, or the file has no rows |
| Prospect ID | Each row | Blank, repeated within the file, or already in the database |
| Numeric fields | `TotalVisits`, `Total Time Spent on Website`, `Page Views Per Visit` | Blank, not a number, or negative |
| Categorical fields | The other 11 attributes | Blank, `Select` (placeholder for "not chosen"), or a value the model never saw in training |

Rejecting a lead, instead of guessing a value, means the model only scores data it can interpret correctly. Each rejected row is reported with the reason, so it can be corrected and resubmitted.

In [ ]:
NUMERIC_COLS = ['TotalVisits', 'Total Time Spent on Website', 'Page Views Per Visit']
CATEGORICAL_COLS = ['Lead Origin', 'Lead Source', 'Do Not Email', 'Do Not Call', 'Last Activity',
                    'Country', 'Specialization', 'How did you hear about X Education',
                    'What is your current occupation', 'City', 'Last Notable Activity']
FEATURE_INPUT_COLS = ['Lead Origin', 'Lead Source', 'Do Not Email', 'Do Not Call',
                      'TotalVisits', 'Total Time Spent on Website', 'Page Views Per Visit',
                      'Last Activity', 'Country', 'Specialization',
                      'How did you hear about X Education', 'What is your current occupation',
                      'City', 'Last Notable Activity']
REQUIRED_COLS = ['Prospect ID'] + FEATURE_INPUT_COLS

# Category values the model learned from, taken from the training data
ALLOWED_VALUES = {col: set(reference[col].dropna().astype(str)) for col in CATEGORICAL_COLS}
EXISTING_IDS = set(reference['prospect_id'].astype(str))

# The exact 132 encoded feature columns, in the order the model expects
scaler = model.named_steps['scaler']
if hasattr(scaler, 'feature_names_in_'):
    MODEL_FEATURES = list(scaler.feature_names_in_)
else:
    ref_X = reference[FEATURE_INPUT_COLS]
    MODEL_FEATURES = list(pd.get_dummies(ref_X, drop_first=True).columns)
print("Model expects", len(MODEL_FEATURES), "encoded features")


class InputFileError(Exception):
    """Raised when the whole file is unusable (missing columns or no rows)."""


def validate(batch):
    """Split a raw batch of leads into valid rows and rejected rows (with reasons)."""
    missing = [c for c in REQUIRED_COLS if c not in batch.columns]
    if missing:
        raise InputFileError("The file is missing required column(s): " + ", ".join(missing))
    if len(batch) == 0:
        raise InputFileError("The file contains no lead rows.")

    batch = batch.copy().reset_index(drop=True)
    for col in REQUIRED_COLS:
        batch[col] = batch[col].fillna('').astype(str).str.strip()

    reasons = [[] for _ in range(len(batch))]

    ids = batch['Prospect ID']
    repeated = ids.duplicated(keep='first') & (ids != '')
    for i, pid in enumerate(ids):
        if pid == '':
            reasons[i].append("Prospect ID is blank")
        elif repeated[i]:
            reasons[i].append(f"Prospect ID {pid} appears more than once in the file")
        elif pid in EXISTING_IDS:
            reasons[i].append(f"Prospect ID {pid} already exists in the database")

    for col in NUMERIC_COLS:
        values = pd.to_numeric(batch[col], errors='coerce')
        for i, (raw, num) in enumerate(zip(batch[col], values)):
            if raw == '':
                reasons[i].append(f"{col} is missing")
            elif pd.isna(num):
                reasons[i].append(f"{col} must be a number (got '{raw}')")
            elif num < 0:
                reasons[i].append(f"{col} cannot be negative (got {raw})")

    for col in CATEGORICAL_COLS:
        for i, raw in enumerate(batch[col]):
            if raw == '' or raw == 'Select':
                reasons[i].append(f"{col} is missing")
            elif raw not in ALLOWED_VALUES[col]:
                reasons[i].append(f"{col} has an unknown value '{raw}'")

    batch['Reason'] = ['; '.join(r) for r in reasons]
    is_valid = batch['Reason'] == ''
    valid = batch[is_valid].drop(columns='Reason').reset_index(drop=True)
    rejected = batch[~is_valid].reset_index(drop=True)
    for col in NUMERIC_COLS:
        valid[col] = valid[col].astype(float)
    return valid, rejected


def assign_tier(prob):
    """FR-04 priority tiers, identical to the training notebook."""
    if prob >= 0.70:
        return 'High'
    elif prob >= 0.40:
        return 'Medium'
    return 'Low'


def encode(valid):
    """One-hot encode and line the columns up exactly with the model's 132 features."""
    encoded = pd.get_dummies(valid[FEATURE_INPUT_COLS]).astype(float)
    return encoded.reindex(columns=MODEL_FEATURES, fill_value=0.0)


def score(valid):
    """Add conversion_probability and priority_tier to the valid leads."""
    scored = valid.copy()
    scored['conversion_probability'] = model.predict_proba(encode(valid))[:, 1]
    scored['priority_tier'] = scored['conversion_probability'].apply(assign_tier)
    return scored

print("Validation rules ready")

## 4. Prepare the input file
By default the notebook builds a **demonstration file**, `new_leads_demo.csv`, so the workflow can be run without preparing data by hand:
- **6 valid leads** (`NEW-0001` to `NEW-0006`), built from the profiles of two existing High, two Medium and two Low leads, with new Prospect IDs
- **8 invalid leads** (`NEW-0007` onward), each with one deliberate problem, to show that bad input is caught

To score your own file instead, set `USE_DEMO_FILE = False` and upload a CSV with the same columns.

In [ ]:
USE_DEMO_FILE = True

def build_demo_batch():
    picks = pd.concat([
        reference[reference['priority_tier'] == tier].sample(2, random_state=42)
        for tier in ['High', 'Medium', 'Low']
    ])
    good = picks[FEATURE_INPUT_COLS].astype(str).reset_index(drop=True)
    for col in NUMERIC_COLS:
        good[col] = picks[col].astype(float).map(lambda v: f"{v:g}").values
    good.insert(0, 'Prospect ID', [f"NEW-{n:04d}" for n in range(1, 7)])

    def broken(pid, col, value):
        row = good.iloc[[0]].copy()
        row['Prospect ID'] = pid
        row[col] = value
        return row

    bad = pd.concat([
        broken('NEW-0007', 'TotalVisits', ''),                        # missing number
        broken('NEW-0008', 'Total Time Spent on Website', '-45'),     # negative number
        broken('NEW-0009', 'TotalVisits', 'five'),                    # text instead of a number
        broken('NEW-0010', 'Lead Source', 'TikTok'),                  # category never seen in training
        broken('NEW-0011', 'Specialization', 'Select'),               # placeholder meaning "not chosen"
        broken('NEW-0001', 'City', good.loc[0, 'City']),              # Prospect ID repeated in the file
        broken(str(reference['prospect_id'].iloc[0]), 'City', good.loc[0, 'City']),  # already in database
        broken('', 'City', good.loc[0, 'City']),                      # blank Prospect ID
    ])
    return pd.concat([good, bad], ignore_index=True)

if USE_DEMO_FILE:
    demo = build_demo_batch()
    demo.to_csv('new_leads_demo.csv', index=False)
    input_file = 'new_leads_demo.csv'
    print("Created new_leads_demo.csv with", len(demo), "rows")
else:
    print("Upload your new leads CSV")
    uploaded = files.upload()
    input_file = list(uploaded.keys())[0]

new_leads = pd.read_csv(input_file, dtype=str, keep_default_na=False)
new_leads

## 5. Validate and score

In [ ]:
try:
    valid, rejected = validate(new_leads)
except InputFileError as e:
    valid, rejected = None, None
    print("FILE REJECTED:", e)

if valid is not None:
    print(f"Accepted: {len(valid)}   Rejected: {len(rejected)}")
    if len(rejected):
        print("\nRejected leads:")
        display(rejected[['Prospect ID', 'Reason']])
    if len(valid):
        scored = score(valid)
        print("\nScored leads:")
        display(scored[['Prospect ID', 'Lead Source', 'Last Activity',
                        'conversion_probability', 'priority_tier']].round({'conversion_probability': 4}))
    else:
        scored = valid.iloc[0:0]
        print("No valid leads to score.")

## 6. Generate the SQL to load the new leads into MySQL
Only accepted leads are written. For each lead, the script inserts a row into `leads` and then a matching row into `conversion_predictions`, linked through `LAST_INSERT_ID()`. `converted` is left empty (NULL) because the outcome of a new lead is not known yet.

The script first deletes any earlier copy of these Prospect IDs, so running it twice does not create duplicates.

**Next:** run `load_new_leads.sql` in MySQL Workbench, then click **Refresh** in Power BI.

In [ ]:
def q(v):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return "NULL"
    return "'" + str(v).replace("\\", "\\\\").replace("'", "''") + "'"

if valid is not None and len(scored):
    id_list = ", ".join(q(p) for p in scored['Prospect ID'])
    lines = [
        "USE salesmesh;",
        "SET SQL_SAFE_UPDATES = 0;",
        f"DELETE cp FROM conversion_predictions cp JOIN leads l ON cp.lead_id = l.lead_id WHERE l.prospect_id IN ({id_list});",
        f"DELETE FROM leads WHERE prospect_id IN ({id_list});",
        "SET SQL_SAFE_UPDATES = 1;",
    ]
    for _, r in scored.iterrows():
        lines.append(
            "INSERT INTO leads (prospect_id, lead_source, lead_origin, occupation, city, "
            "total_visits, time_on_website, last_activity, converted) VALUES "
            f"({q(r['Prospect ID'])}, {q(r['Lead Source'])}, {q(r['Lead Origin'])}, "
            f"{q(r['What is your current occupation'])}, {q(r['City'])}, {int(round(r['TotalVisits']))}, "
            f"{int(round(r['Total Time Spent on Website']))}, {q(r['Last Activity'])}, NULL);"
        )
        lines.append(
            "INSERT INTO conversion_predictions (lead_id, campaign_id, user_id, "
            "conversion_probability, priority_tier) VALUES "
            f"(LAST_INSERT_ID(), NULL, NULL, {round(float(r['conversion_probability']), 6)}, {q(r['priority_tier'])});"
        )
    lines += [
        "SELECT l.lead_id, l.prospect_id, cp.conversion_probability, cp.priority_tier, cp.predicted_at "
        f"FROM leads l JOIN conversion_predictions cp ON cp.lead_id = l.lead_id WHERE l.prospect_id IN ({id_list});",
        "SELECT COUNT(*) AS total_leads FROM leads;",
    ]
    with open('load_new_leads.sql', 'w', encoding='utf-8') as f:
        f.write("\n".join(lines))
    print(f"Created load_new_leads.sql ({len(scored)} leads)")
else:
    print("No SQL created: there are no valid leads to load.")

## 7. Automated test cases
These tests check expected behaviour, invalid input and failure conditions. They always run on the demonstration batch, so the results are repeatable. Each result is produced by actually running the code, not typed in by hand.

In [ ]:
test_rows = []

def record(test_id, case, expected, actual, passed):
    test_rows.append({'Test ID': test_id, 'Test case': case, 'Expected result': expected,
                      'Actual result': actual, 'Status': 'Pass' if passed else 'Fail',
                      'Corrective action': 'None required' if passed else 'To investigate'})

def reason_for(rej, pid, position=0):
    match = rej[rej['Prospect ID'] == pid]
    return match['Reason'].iloc[position] if len(match) > position else '(not rejected)'

t_batch = build_demo_batch()
t_valid, t_rejected = validate(t_batch)
t_scored = score(t_valid)

# Expected behaviour
record('T01', 'Batch with 6 valid and 8 invalid leads', '6 accepted, 8 rejected',
       f"{len(t_valid)} accepted, {len(t_rejected)} rejected", len(t_valid) == 6 and len(t_rejected) == 8)
record('T02', 'Encoded input matches the model', f"{len(MODEL_FEATURES)} features, same order as training",
       f"{encode(t_valid).shape[1]} features, order {'matches' if list(encode(t_valid).columns) == MODEL_FEATURES else 'differs'}",
       list(encode(t_valid).columns) == MODEL_FEATURES)
p = t_scored['conversion_probability']
record('T03', 'Probabilities are valid', 'Every probability between 0 and 1',
       f"min {p.min():.4f}, max {p.max():.4f}", bool(((p >= 0) & (p <= 1)).all()))
boundary = {0.70: 'High', 0.6999: 'Medium', 0.40: 'Medium', 0.3999: 'Low'}
got = {k: assign_tier(k) for k in boundary}
record('T04', 'Tier thresholds at the boundaries (FR-04)', '0.70 High, 0.6999 Medium, 0.40 Medium, 0.3999 Low',
       ", ".join(f"{k} {v}" for k, v in got.items()), got == boundary)
record('T05', 'Every scored lead gets the right tier', 'Tier matches its probability for all 6 leads',
       f"{(t_scored['priority_tier'] == p.apply(assign_tier)).sum()} of {len(t_scored)} match",
       bool((t_scored['priority_tier'] == p.apply(assign_tier)).all()))
again = score(t_valid)['conversion_probability']
record('T06', 'Scoring the same leads twice', 'Identical probabilities both times',
       'Identical' if np.allclose(p, again) else 'Different', bool(np.allclose(p, again)))

# Invalid input
invalid_cases = [
    ('T07', 'NEW-0007', 'Missing TotalVisits', 'TotalVisits is missing'),
    ('T08', 'NEW-0008', 'Negative time on website (-45)', 'cannot be negative'),
    ('T09', 'NEW-0009', "Text instead of a number ('five')", 'must be a number'),
    ('T10', 'NEW-0010', "Unknown lead source ('TikTok')", 'unknown value'),
    ('T11', 'NEW-0011', "Placeholder 'Select' in Specialization", 'Specialization is missing'),
    ('T12', 'NEW-0001', 'Prospect ID repeated in the same file', 'appears more than once'),
    ('T13', str(reference['prospect_id'].iloc[0]), 'Prospect ID already in the database', 'already exists'),
    ('T14', '', 'Blank Prospect ID', 'Prospect ID is blank'),
]
for test_id, pid, case, expected_text in invalid_cases:
    actual = reason_for(t_rejected, pid)
    record(test_id, case, f"Rejected: '{expected_text}'", f"Rejected: {actual}" if actual != '(not rejected)' else actual,
           expected_text in actual)

# Failure conditions
try:
    validate(t_batch.drop(columns=['TotalVisits']))
    record('T15', 'File missing a required column', 'Whole file rejected with a clear message', 'File was accepted', False)
except InputFileError as e:
    record('T15', 'File missing a required column', 'Whole file rejected with a clear message', str(e), True)
except Exception as e:
    record('T15', 'File missing a required column', 'Whole file rejected with a clear message', f"Crashed: {e!r}", False)

try:
    validate(t_batch.iloc[0:0])
    record('T16', 'Empty file (headers only)', 'Whole file rejected with a clear message', 'File was accepted', False)
except InputFileError as e:
    record('T16', 'Empty file (headers only)', 'Whole file rejected with a clear message', str(e), True)
except Exception as e:
    record('T16', 'Empty file (headers only)', 'Whole file rejected with a clear message', f"Crashed: {e!r}", False)

test_results = pd.DataFrame(test_rows)
test_results.to_csv('test_results_scoring.csv', index=False)
print(f"{(test_results['Status'] == 'Pass').sum()} of {len(test_results)} tests passed")
pd.set_option('display.max_colwidth', None)
test_results

## 8. Download the outputs
| File | Folder |
|---|---|
| new_leads_demo.csv | Data/ |
| scored_new_leads.csv, rejected_leads.csv | predictions/ |
| load_new_leads.sql | database/ |
| test_results_scoring.csv | report/ |

In [ ]:
outputs = []
if USE_DEMO_FILE:
    outputs.append('new_leads_demo.csv')
if valid is not None:
    scored.to_csv('scored_new_leads.csv', index=False)
    rejected.to_csv('rejected_leads.csv', index=False)
    outputs += ['scored_new_leads.csv', 'rejected_leads.csv']
    if len(scored):
        outputs.append('load_new_leads.sql')
outputs.append('test_results_scoring.csv')

for f in outputs:
    files.download(f)
print("Downloaded:", ", ".join(outputs))